<a href="https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyushKhatri-Dev/flyrank-search-intelligence/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Lane confirmed: Lane 4 — CTR / Engagement Opportunity Scoring.** No switch.

### Signal check 1 — CTR vs position (behind FlyRank's CTR-fix logic): **CONFIRMED**

| Position tier | n | Median CTR | Zero-click pages |
|---|---:|---:|---:|
| 1-3 | 8,162 | 0.211 | 21.6% |
| 4-10 | 43,764 | 0.183 | 27.8% |
| 11-20 | 18,149 | 0.094 | 43.2% |

CTR falls with position and the share of pages that got no clicks at all rises
sharply — 43% of pages in tier 11-20 received zero clicks despite 100+
impressions. This is the premise my whole lane rests on: comparing a page's CTR
against pages at a similar position is meaningful, comparing it against the
whole slice is not.

### Signal check 2 — staleness (behind FlyRank's refresh flags): **FALSE**

| Staleness bucket | n | Median CTR |
|---|---:|---:|
| <30d | 69,989 | 0.167 |
| 30-90d | 26 | 0.000 |
| 90-180d | 43 | 0.217 |
| 180-365d | 17 | 0.136 |
| 365d+ | 0 | — |

Almost every page lands in one bucket, so the signal has no spread to test. The
distribution shows why: `days_since_update` runs from -127 to +234, and **56,874
of 70,075 pages are negative** — updated *after* my window opened. The single
most common value is +4 days, shared by 12,734 pages.

That is not a content update history. It reads like a bulk sync or optimisation
run stamped across thousands of pages at once.

Two conclusions, and the second matters more. Staleness cannot be measured on
this slice at all. And `content_updated_date` mostly sits **after** my decision
moment, so using it would have fed future information into a rule meant to be
knowable at the decision moment. The check saved the rule rather than just
failing it.

### The rule, in plain words

For every page with enough exposure, compare its CTR against the median CTR of
pages in its own position tier. If it sits below that median, estimate how many
clicks it plausibly missed — the shortfall in CTR multiplied by the impressions
it actually received. Rank pages by that estimate.

Multiplying by impressions is deliberate. In ML-02 I ranked on the raw gap and
all fifty top pages came from one tier, because an absolute difference from a
tier median is biased toward whichever tier has the highest median. Scoring in
estimated missed clicks fixes that and puts the score in a unit a reviewer
understands: "this page plausibly lost about N clicks this month."

- **Score:** `estimated_missed_clicks = impressions × (tier_median_ctr − ctr) / 100`, floored at 0
- **Reason code:** `below_tier_ctr` — the page's CTR is below the median for its position tier
- **Action label:** `review_title_and_meta`

Both inputs are knowable at the decision moment: impressions and position are
measurements taken during the window, and the tier median is computed from the
same window. Nothing from a future window and no product flag enters the score.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip install -q duckdb huggingface_hub

import os, json
import duckdb, pandas as pd, numpy as np
from google.colab import userdata
from huggingface_hub import login

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE OR REPLACE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH, WINDOW_START = "2026-03", "2026-03-01"
DAILY = f"read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/data_0.parquet')"
CONTENT = f"read_parquet('{BASE}/dim_content.parquet')"

frame = con.sql(f"""
    WITH agg AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS impressions,
               SUM(gsc_clicks) AS clicks,
               SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions) AS avg_position
        FROM {DAILY}
        WHERE gsc_data_available IS TRUE
        GROUP BY 1, 2
    )
    SELECT a.*, c.content_type, c.main_intent, c.word_count,
           DATE_DIFF('day', c.content_created_date, DATE '{WINDOW_START}') AS content_age_days,
           DATE_DIFF('day', c.content_updated_date, DATE '{WINDOW_START}') AS days_since_update,
           a.clicks * 100.0 / a.impressions AS ctr
    FROM agg a
    JOIN {CONTENT} c USING (client_hash_id, content_hash_id)
    WHERE a.impressions >= 100
      AND a.avg_position BETWEEN 1 AND 20
      AND DATE_DIFF('day', c.content_created_date, DATE '{WINDOW_START}') >= 0
""").df()

print(f"Frame: {len(frame):,} pages, {frame['client_hash_id'].nunique()} clients\n")

# ---------- SIGNAL 1: CTR vs position (behind FlyRank's CTR-fix logic) ----------
frame["position_tier"] = pd.cut(frame["avg_position"], [0, 3, 10, 20],
                                labels=["1-3", "4-10", "11-20"])
sig1 = frame.groupby("position_tier", observed=True).agg(
    n=("ctr", "size"), median_ctr=("ctr", "median"), mean_ctr=("ctr", "mean"),
    pct_zero_clicks=("clicks", lambda s: (s == 0).mean() * 100)).round(3)
print("SIGNAL 1 — CTR by position tier (CTR-fix logic)")
print(sig1.to_string(), "\n")

# ---------- SIGNAL 2: staleness (behind FlyRank's refresh flags)
frame["staleness_bucket"] = pd.cut(frame["days_since_update"],
                                   [-np.inf, 30, 90, 180, 365, np.inf],
                                   labels=["<30d", "30-90d", "90-180d", "180-365d", "365d+"])
sig2 = frame.groupby("staleness_bucket", observed=True).agg(
    n=("ctr", "size"), median_ctr=("ctr", "median"),
    median_position=("avg_position", "median"),
    median_impressions=("impressions", "median")).round(3)
print("SIGNAL 2 — CTR by staleness bucket (refresh-flag logic)")
print(sig2.to_string())

d = frame["days_since_update"]
print("days_since_update distribution:")
print(d.describe().round(1).to_string())
print(f"\nNegative (updated AFTER window start): {(d < 0).sum():,} of {len(d):,}")
print(f"Zero or positive:                      {(d >= 0).sum():,}")
print("\nTop 10 most common values:")
print(d.value_counts().head(10).to_string())


KeyboardInterrupt: 

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

The score is:

estimated_missed_clicks = impressions × (tier_median_ctr − ctr) / 100


floored at zero, so pages already above their tier median score 0 and drop out.

I score in estimated clicks rather than in raw CTR gap on purpose. In ML-02 I
ranked on the raw gap and all 50 top pages came from one tier, because an
absolute difference from a tier median favours whichever tier has the highest
median. Multiplying by impressions also puts the score in a unit a reviewer
understands: roughly how many clicks this page plausibly missed this month.

Every row in the queue carries:

- **reason code** `below_tier_ctr` — CTR is under the median for its position tier
- **action** `review_title_and_meta`
- **confidence** from volume: high at 1,000+ impressions, medium at 300+, low below that

The queue is written to `work/outputs/baseline_action_score.csv`, sorted by score,
with a rank column. That file stays out of git by design — the notebook rebuilds
it on every run. The run's metrics go to `work/outputs/w04_baseline_metrics.json`.

**Result:** 35,036 of 70,075 pages flagged (50%). The top 50 breaks down as 39
pages from tier 4-10, 11 from tier 1-3, and none from 11-20. Section 4 covers
why that last number is a problem.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os

# --- tier median from the same window, no future information ---
frame["tier_median_ctr"] = frame.groupby("position_tier", observed=True)["ctr"].transform("median")
frame["ctr_shortfall"] = (frame["tier_median_ctr"] - frame["ctr"]).clip(lower=0)
frame["estimated_missed_clicks"] = frame["impressions"] * frame["ctr_shortfall"] / 100

queue = frame[frame["estimated_missed_clicks"] > 0].copy()
queue["reason_code"] = "below_tier_ctr"
queue["action"] = "review_title_and_meta"
queue["confidence"] = np.where(queue["impressions"] >= 1000, "high",
                       np.where(queue["impressions"] >= 300, "medium", "low"))

queue = queue.sort_values("estimated_missed_clicks", ascending=False).reset_index(drop=True)
queue["rank"] = queue.index + 1

cols = ["rank", "client_hash_id", "content_hash_id", "avg_position", "position_tier",
        "impressions", "clicks", "ctr", "tier_median_ctr", "ctr_shortfall",
        "estimated_missed_clicks", "reason_code", "action", "confidence"]

os.makedirs("work/outputs", exist_ok=True)
queue[cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

metrics = {
    "month": MONTH,
    "pages_in_frame": int(len(frame)),
    "clients": int(frame["client_hash_id"].nunique()),
    "pages_flagged": int(len(queue)),
    "pct_flagged": round(len(queue) / len(frame) * 100, 1),
    "tier_medians": {str(k): round(float(v), 3)
                     for k, v in frame.groupby("position_tier", observed=True)["ctr"].median().items()},
    "top50_tier_mix": {str(k): int(v)
                       for k, v in queue.head(50)["position_tier"].value_counts().items()},
    "signal_verdicts": {"ctr_vs_position": "CONFIRMED", "staleness": "FALSE"},
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Flagged {len(queue):,} of {len(frame):,} pages ({metrics['pct_flagged']}%)")
print("\nTier mix in top 50 (the ML-02 bias check):")
print(queue.head(50)["position_tier"].value_counts().to_string())
print("\nTop 10:")
print(queue.head(10)[["rank", "position_tier", "avg_position", "impressions",
                      "ctr", "tier_median_ctr", "estimated_missed_clicks", "confidence"]]
      .round(3).to_string(index=False))

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

All twenty rows have the same reason code, the same action, and high confidence
on volume. What changes is how much I trust each one. I see three groups.

**Standard cases, 8 of 20.** A real gap against the tier median, with enough
volume that the gap isn't noise. Rank 2 is the clearest: position 3.17, 143,019
impressions, 43 clicks, so CTR 0.030 against a tier median of 0.183. These are
what the rule was built for. What would make them wrong: the query may not match
what the page actually offers, and a title rewrite doesn't fix that.

**Near-zero CTR with heavy exposure, 7 of 20.** Rank 1 is at position 2.69 with
134,984 impressions and 1 click. Rank 17 is at position 1.40 with 44,707
impressions and 0 clicks. A page ranking that high with no clicks is probably not
a title problem. More likely a tracking gap, or a SERP feature answering the
query on the results page. What would make them wrong: almost anything. They rank
top because their gap is biggest, which is the weak spot of a shortfall score.
The worst data ends up on top instead of the best page to fix. A reviewer should
check these before editing anything.

**Already at position 1-3, 7 of 20 (overlaps with the group above).** Ranks 6, 8,
12, 16 and 19. Low CTR at top-3 usually means the query is informational and gets
answered on the results page. What would make them wrong: the ceiling is low.
Even a good rewrite recovers less than the score suggests.

One more thing I noticed: every page in the top 20 is `keyword article`. So
content type tells me nothing at this end of the queue.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20 = queue.head(20).copy()

top20["risk"] = np.where(
    top20["ctr"] < 0.01, "near-zero CTR despite heavy exposure — check tracking or SERP feature",
    np.where(top20["avg_position"] <= 3, "already top-3; low CTR here may be intent, not metadata",
    np.where(top20["ctr_shortfall"] < 0.05, "small shortfall, ranked high only on volume",
             "standard case — gap and volume both meaningful")))

print(top20[["rank", "position_tier", "avg_position", "impressions", "clicks", "ctr",
             "estimated_missed_clicks", "content_type", "main_intent", "risk"]]
      .round(3).to_string(index=False))

print("\nRisk pattern counts in top 20:")
print(top20["risk"].value_counts().to_string())
print(f"\nPages in top 20 with CTR < 0.01: {(top20['ctr'] < 0.01).sum()}")
print(f"Pages in top 20 already at position <= 3: {(top20['avg_position'] <= 3).sum()}")

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### The bias that's still there

| Tier | Flagged | In top 50 |
|---|---:|---:|
| 1-3 | 4,080 | 11 |
| 4-10 | 21,882 | 39 |
| 11-20 | 9,074 | **0** |

Tier 11-20 has 9,074 flagged pages and none of them reach the top 50. I expected
some to show up, since 43% of that tier gets zero clicks, so this surprised me.

It isn't random. My score is impressions times shortfall, and tier 11-20 is
smaller on both. Its median CTR is 0.094, so the biggest shortfall a page there
can have is 0.094, against 0.211 for tier 1-3. Its median impressions are 581,
against 1,849 for tier 1-3. Both numbers are smaller and I multiply them, so
deep-ranked pages basically can't outscore shallow ones.

That's a problem, because tier 11-20 is where the worst outcomes are. In ML-02
the bias went the other way: an absolute gap put all 50 top pages in tier 4-10.
Weighting by impressions changed which end it favours, it didn't remove the
trade-off. One score, one axis.

Two things I can test in the model week: score the shortfall as a fraction of the
tier median so tiers compete evenly, or keep the click-based score but rank
within each tier instead of globally.

### Leakage check

| Input | Knowable because |
|---|---|
| `impressions` | summed over the window, a measurement of exposure already received |
| `avg_position` | impression-weighted over the same window |
| `ctr` | the thing being scored, not a predictor of itself |
| `tier_median_ctr` | computed from the same window, from the same pages |

Excluded on purpose:

- `days_since_update` and `content_updated_date`. 56,874 of 70,075 values fall
  after my window start, so using them would put future information into a rule
  that has to work at the decision moment. Signal check 2 caught this.
- Any FlyRank product flag. Not shipped in this release, so there's nothing to
  accidentally rebuild.

An assertion in the code confirms no future-dated column reached the exported
queue. It passes.

### Coverage limit

The rule flags 35,036 of 70,075 pages, exactly 50%. That's just what a median
does, half the pool sits below it. So the flag itself carries no information.
What makes the queue useful is the ordering, not the flag.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- weak picks: which tier never reaches the top of the queue, and why ---
print("Tier mix — top 50 vs the flagged pool:")
print(pd.concat([
    queue.head(50)["position_tier"].value_counts().rename("top_50"),
    queue["position_tier"].value_counts().rename("all_flagged"),
], axis=1).to_string())

print("\nMax possible shortfall by tier (= that tier's median CTR):")
print(frame.groupby("position_tier", observed=True)["ctr"].median().round(3).to_string())

print("\nMedian impressions by tier:")
print(frame.groupby("position_tier", observed=True)["impressions"].median().to_string())

# --- leakage check: every input, and when it was knowable ---
inputs = {
    "impressions": "summed over the window — a measurement, not an outcome",
    "avg_position": "impression-weighted over the same window",
    "ctr": "clicks/impressions over the same window — the quantity being scored, not a predictor",
    "tier_median_ctr": "computed from the same window, from the same pages",
}
print("\nInputs to the score and when each was knowable:")
for k, v in inputs.items():
    print(f"  {k:20s} {v}")

excluded = {
    "days_since_update": "56,874/70,075 values are negative — mostly AFTER the window start",
    "content_updated_date": "same problem; reads as a bulk sync stamp, not an edit history",
    "any FlyRank product flag": "not shipped in this release by design",
}
print("\nDeliberately excluded:")
for k, v in excluded.items():
    print(f"  {k:24s} {v}")

assert "days_since_update" not in cols, "future-dated column leaked into the queue"
print("\nLeakage check passed: no future-window or product-flag input in the score.")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.